# AI-Powered Job Search & Application System
**Capstone Project — LLM Engineering Course**

Multi-agent pipeline: resume parsing → job discovery → match scoring → cover letter generation

**Models:** Groq `llama-3.1-8b-instant` (free tier) + local `all-MiniLM-L6-v2` embeddings  
**Course skills:** Weeks 1–8 (scraping, agents, RAG, OSS models, deployment, UI)

## 1. Setup

In [ ]:
%pip install -q groq chromadb sentence-transformers gradio PyMuPDF python-docx pydantic python-dotenv requests beautifulsoup4

In [ ]:
import os, json, re, io, time, hashlib, urllib.parse
from html import unescape
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
import fitz
from docx import Document as DocxDocument
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pydantic import BaseModel, Field, field_validator
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq
import gradio as gr

load_dotenv()

GROQ_API_KEY  = os.getenv("GROQ_API_KEY", "")
SERPAPI_KEY   = os.getenv("SERPAPI_KEY", "")
LLM_MODEL     = "llama-3.1-8b-instant"
EMBED_MODEL   = "all-MiniLM-L6-v2"
MAX_JOBS      = 20
REQUEST_TIMEOUT = 15

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not set. Add it to a .env file or set it as an environment variable.")

llm = Groq(api_key=GROQ_API_KEY)

## 2. Data Models

In [ ]:
class JobPosting(BaseModel):
    title: str
    company: str
    location: str
    description: str
    url: str
    salary: str = ""
    posted_date: str = ""
    source: str = ""

    @property
    def uid(self) -> str:
        return hashlib.md5(self.url.encode()).hexdigest()[:8]


class Experience(BaseModel):
    title: str = ""
    company: str = ""
    dates: str = ""
    bullets: list[str] = Field(default_factory=list)


class UserProfile(BaseModel):
    name: str = "Candidate"
    email: str = ""
    phone: str = ""
    skills: list[str] = Field(default_factory=list)
    experiences: list[Experience] = Field(default_factory=list)
    education: list[str] = Field(default_factory=list)
    achievements: list[str] = Field(default_factory=list)
    raw_text: str = ""


class ScoredJob(BaseModel):
    job: JobPosting
    score: int = Field(ge=0, le=100)
    reasons: list[str] = Field(default_factory=list)
    skill_gaps: list[str] = Field(default_factory=list)
    recommendation: str = ""

    @field_validator("score", mode="before")
    @classmethod
    def clamp(cls, v):
        try:
            return max(0, min(100, int(v)))
        except (TypeError, ValueError):
            return 0

## 3. LLM Utilities

In [ ]:
def chat(system: str, user: str, temperature: float = 0.3, max_tokens: int = 1500) -> str:
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
    )
    return response.choices[0].message.content.strip()


def chat_json(system: str, user: str, retries: int = 2) -> dict:
    system_json = system + "\n\nRespond ONLY with valid JSON. No markdown, no explanation."
    for attempt in range(retries + 1):
        try:
            raw = chat(system_json, user, temperature=0.1)
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if not match:
                raise ValueError("No JSON object found")
            return json.loads(match.group())
        except (json.JSONDecodeError, ValueError):
            if attempt == retries:
                return {}
            time.sleep(1)
    return {}

## 4. Vector Store (RAG)

In [ ]:
_embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)


def _fresh_collections():
    client = chromadb.Client()
    for name in ("jobs", "profile"):
        try:
            client.delete_collection(name)
        except Exception:
            pass
    job_col     = client.create_collection("jobs",    embedding_function=_embed_fn)
    profile_col = client.create_collection("profile", embedding_function=_embed_fn)
    return client, job_col, profile_col


def index_profile(profile_col, profile: UserProfile) -> None:
    text = f"{profile.name}\nSkills: {', '.join(profile.skills)}\n"
    text += "\n".join(f"{e.title} at {e.company}" for e in profile.experiences)
    text += "\n" + "\n".join(profile.achievements)
    profile_col.upsert(ids=["user_profile"], documents=[text])


def index_jobs(job_col, jobs: list[JobPosting]) -> None:
    if not jobs:
        return
    job_col.upsert(
        ids=[f"job_{i}" for i in range(len(jobs))],
        documents=[f"{j.title} {j.company} {j.description}" for j in jobs],
        metadatas=[{"title": j.title, "company": j.company, "url": j.url, "idx": i} for i, j in enumerate(jobs)],
    )


def retrieve_top_matches(profile: UserProfile, jobs: list[JobPosting], top_n: int = 10) -> list[JobPosting]:
    if not jobs:
        return []
    _, job_col, profile_col = _fresh_collections()
    index_profile(profile_col, profile)
    index_jobs(job_col, jobs)

    n = min(top_n, len(jobs))
    profile_doc = profile_col.get(ids=["user_profile"])["documents"][0]
    results = job_col.query(query_texts=[profile_doc], n_results=n)

    matched_indices = {m["idx"] for m in results["metadatas"][0]}
    return [j for i, j in enumerate(jobs) if i in matched_indices]

## 5. Agent 1 — Resume Parser

In [ ]:
def _extract_text_from_pdf(path: str) -> str:
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def parse_resume(file_path: str) -> UserProfile:
    raw = _extract_text_from_pdf(file_path)

    schema = {
        "name": "string",
        "email": "string",
        "phone": "string",
        "skills": ["string"],
        "experiences": [{"title": "string", "company": "string", "dates": "string", "bullets": ["string"]}],
        "education": ["string — degree, institution, year"],
        "achievements": ["string — quantified accomplishments"],
    }

    result = chat_json(
        system=f"Extract structured resume data. Output schema: {json.dumps(schema)}",
        user=f"Resume text:\n{raw[:4000]}",
    )

    experiences = [
        Experience(**e) if isinstance(e, dict) else Experience()
        for e in result.get("experiences", [])
    ]

    profile = UserProfile(
        name=result.get("name", "Candidate"),
        email=result.get("email", ""),
        phone=result.get("phone", ""),
        skills=result.get("skills", []),
        experiences=experiences,
        education=result.get("education", []),
        achievements=result.get("achievements", []),
        raw_text=raw,
    )
    return profile

## 6. Agent 2 — Job Scraper

In [ ]:
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
}

def _clean(text: str) -> str:
    if not text:
        return ""
    from html import unescape
    prev = None
    while prev != text:
        prev = text
        text = unescape(text)
    return text

def _safe_get(url: str) -> Optional[requests.Response]:
    try:
        r = requests.get(url, headers=_HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r
    except requests.RequestException:
        return None


def _scrape_jobicy(query: str) -> list[JobPosting]:
    tag = query.strip().split()[0].lower()
    r = _safe_get(f"https://jobicy.com/api/v2/remote-jobs?count=20&tag={urllib.parse.quote(tag)}")
    if not r:
        return []
    jobs = []
    for item in r.json().get("jobs", []):
        jobs.append(JobPosting(
            title=_clean(item.get("jobTitle", "")),
            company=_clean(item.get("companyName", "")),
            location=item.get("jobGeo", "Remote"),
            description=_clean(item.get("jobExcerpt", ""))[:1000],
            url=item.get("url", ""),
            salary=str(item.get("annualSalaryMin", "")),
            posted_date=item.get("pubDate", "")[:10],
            source="Jobicy",
        ))
    return jobs


def _scrape_remoteok(query: str) -> list[JobPosting]:
    r = _safe_get("https://remoteok.com/api")
    if not r:
        return []
    q = query.lower()
    jobs = []
    for item in r.json():
        if not isinstance(item, dict) or "id" not in item:
            continue
        text = (item.get("position", "") + " " + " ".join(item.get("tags", []))).lower()
        if not any(word in text for word in q.split()):
            continue
        jobs.append(JobPosting(
            title=_clean(item.get("position", "")),
            company=_clean(item.get("company", "")),
            location=item.get("location", "Remote") or "Remote",
            description=_clean(item.get("description", ""))[:1000],
            url=item.get("url", f"https://remoteok.com/l/{item.get('id', '')}"),
            salary=str(item.get("salary", "")),
            posted_date=str(item.get("date", ""))[:10],
            source="RemoteOK",
        ))
        if len(jobs) >= 15:
            break
    return jobs


def _scrape_themuse(query: str) -> list[JobPosting]:
    params = {"page": 1, "descending": "true", "query": query}
    r = _safe_get(f"https://www.themuse.com/api/public/jobs?{urllib.parse.urlencode(params)}")
    if not r:
        return []
    jobs = []
    for item in r.json().get("results", []):
        locations = item.get("locations", [{}])
        location = locations[0].get("name", "Remote") if locations else "Remote"
        levels = item.get("levels", [{}])
        company = item.get("company", {}).get("name", "")
        url = item.get("refs", {}).get("landing_page", "")
        jobs.append(JobPosting(
            title=_clean(item.get("name", "")),
            company=_clean(company),
            description=_clean(item.get("contents", "") or "")[:1000],
            location=location,
            url=url,
            salary=levels[0].get("name", "") if levels else "",
            posted_date=item.get("publication_date", "")[:10],
            source="TheMuse",
        ))
    return jobs



def _scrape_serpapi(query: str, location: str) -> list[JobPosting]:
    if not SERPAPI_KEY:
        return []
    params = {
        "engine": "google_jobs",
        "q": query,
        "location": location,
        "api_key": SERPAPI_KEY,
        "num": MAX_JOBS,
    }
    r = _safe_get(f"https://serpapi.com/search?{urllib.parse.urlencode(params)}")
    if not r:
        return []
    jobs = []
    for item in r.json().get("jobs_results", []):
        jobs.append(JobPosting(
            title=item.get("title", ""),
            company=item.get("company_name", ""),
            location=item.get("location", ""),
            description=item.get("description", "")[:1000],
            url=item.get("share_link", ""),
            salary=str(item.get("detected_extensions", {}).get("salary", "")),
            posted_date=item.get("detected_extensions", {}).get("posted_at", ""),
            source="Google Jobs",
        ))
    return jobs


def scrape_jobs(query: str, location: str = "Remote", remote_only: bool = True) -> list[JobPosting]:
    sources = {
        "Jobicy":   _scrape_jobicy(query),
        "RemoteOK": _scrape_remoteok(query),
        "TheMuse":  _scrape_themuse(query),
        "SerpAPI":  _scrape_serpapi(query, location),
    }

    diagnostics = {name: len(jobs) for name, jobs in sources.items()}
    print(f"Scraper results: {diagnostics}")

    all_jobs = [j for jobs in sources.values() for j in jobs]

    seen, unique = set(), []
    for j in all_jobs:
        if j.url and j.url not in seen:
            seen.add(j.url)
            unique.append(j)

    print(f"Total unique jobs after dedup: {len(unique)}")
    return unique[:MAX_JOBS]

## 7. Agent 3 — Job Matcher & Scorer

In [ ]:
def score_job(profile: UserProfile, job: JobPosting) -> ScoredJob:
    schema = {
        "score": "0-100 integer",
        "reasons": ["top 3 reasons this is or is not a good fit"],
        "skill_gaps": ["skills in job description the candidate lacks"],
        "recommendation": "one sentence recommendation",
    }
    result = chat_json(
        system=(
            f"You are a career advisor. Score candidate-job fit 0-100. "
            f"Output schema: {json.dumps(schema)}"
        ),
        user=(
            f"Candidate: {profile.name}\n"
            f"Skills: {', '.join(profile.skills[:15])}\n"
            f"Experience: {' | '.join(e.title + ' at ' + e.company for e in profile.experiences[:3])}\n"
            f"Achievements: {'; '.join(profile.achievements[:2])}\n\n"
            f"Job: {job.title} at {job.company} ({job.location})\n"
            f"Description: {job.description[:600]}"
        ),
    )
    return ScoredJob(
        job=job,
        score=result.get("score", 0),
        reasons=result.get("reasons", []),
        skill_gaps=result.get("skill_gaps", []),
        recommendation=result.get("recommendation", ""),
    )


def match_and_score(profile: UserProfile, jobs: list[JobPosting], top_n: int = 5) -> list[ScoredJob]:
    candidates = retrieve_top_matches(profile, jobs, top_n=top_n * 2)
    scored = [score_job(profile, j) for j in candidates]
    return sorted(scored, key=lambda x: x.score, reverse=True)[:top_n]

## 8. Agent 4 — Cover Letter Generator

In [ ]:
_COVER_LETTER_EXAMPLES = """
Example 1 — opening line:
"When I reduced model inference latency by 40% at Konga, I learned that great ML engineering is as much about systems thinking as it is about algorithms — exactly the mindset your JD describes."

Example 2 — opening line:
"Building a real-time fraud detection pipeline that processed 2M transactions daily taught me more about production ML than any course could — and that experience maps directly to what you're hiring for."
""".strip()


def generate_cover_letter(profile: UserProfile, scored: ScoredJob) -> str:
    job = scored.job
    top_achievement = profile.achievements[0] if profile.achievements else "delivering impactful results"

    system = (
        "You are an expert career coach. Write exactly one tailored 3-paragraph cover letter. "
        "Never write more than one letter. Never add a subject line, date, or second letter. "
        "Never open with 'I am writing to express my interest'. Lead with a specific achievement. "
        f"Style reference for opening lines only:\n{_COVER_LETTER_EXAMPLES}"
    )
    user = (
        f"Candidate: {profile.name}\n"
        f"Email: {profile.email}\n"
        f"Skills: {', '.join(profile.skills[:10])}\n"
        f"Top achievement: {top_achievement}\n"
        f"Recent role: {profile.experiences[0].title + ' at ' + profile.experiences[0].company if profile.experiences else 'N/A'}\n\n"
        f"Applying to: {job.title} at {job.company} ({job.location})\n"
        f"Job description excerpt: {job.description[:500]}\n"
        f"Key fit reasons: {'; '.join(scored.reasons[:2])}"
    )
    return chat(system, user, temperature=0.6, max_tokens=700)


def cover_letter_to_docx(profile: UserProfile, job: JobPosting, letter_text: str) -> bytes:
    doc = DocxDocument()

    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(11)

    for section in doc.sections:
        section.top_margin    = Pt(72)
        section.bottom_margin = Pt(72)
        section.left_margin   = Pt(90)
        section.right_margin  = Pt(90)

    name_para = doc.add_paragraph(profile.name)
    name_para.runs[0].bold = True
    name_para.runs[0].font.size = Pt(14)
    name_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

    contact = " | ".join(filter(None, [profile.email, profile.phone]))
    if contact:
        cp = doc.add_paragraph(contact)
        cp.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph()

    hiring = doc.add_paragraph(f"Hiring Team, {job.company}")
    hiring.runs[0].bold = True

    role = doc.add_paragraph(f"Re: {job.title}")
    role.runs[0].italic = True

    doc.add_paragraph()

    for para in letter_text.strip().split("\n\n"):
        if para.strip():
            doc.add_paragraph(para.strip())

    doc.add_paragraph()
    doc.add_paragraph("Sincerely,")
    closing = doc.add_paragraph(profile.name)
    closing.runs[0].bold = True

    buf = io.BytesIO()
    doc.save(buf)
    return buf.getvalue()

## 9. Orchestrator

In [ ]:
@dataclass
class PipelineResult:
    profile: UserProfile
    jobs_found: int
    scored_jobs: list[ScoredJob]
    cover_letters: dict[str, str]      # job uid -> letter text
    cover_letter_docs: dict[str, bytes] # job uid -> docx bytes


def run_pipeline(
    resume_path: str,
    job_query: str,
    location: str = "Remote",
    remote_only: bool = True,
    top_n: int = 5,
) -> PipelineResult:
    profile = parse_resume(resume_path)
    jobs = scrape_jobs(job_query, location=location, remote_only=remote_only)

    if not jobs:
        return PipelineResult(profile, 0, [], {}, {})

    scored = match_and_score(profile, jobs, top_n=top_n)

    letters, docs = {}, {}
    for s in scored:
        text = generate_cover_letter(profile, s)
        letters[s.job.uid] = text
        docs[s.job.uid]    = cover_letter_to_docx(profile, s.job, text)

    return PipelineResult(
        profile=profile,
        jobs_found=len(jobs),
        scored_jobs=scored,
        cover_letters=letters,
        cover_letter_docs=docs,
    )

## 10. Gradio UI

In [ ]:
_state: dict = {}


def _set_loading():
    return (
        gr.update(value="Searching and scoring jobs — this takes 30–60 seconds...", visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        "",
        gr.update(interactive=False, value="Searching..."),
    )


def _run(resume_file, query, location, remote_only, top_n):
    if resume_file is None:
        return (
            gr.update(value="Upload a resume PDF to continue.", visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            "",
            gr.update(interactive=True, value="Find & Score Jobs"),
        )

    result = run_pipeline(
        resume_path=resume_file.name,
        job_query=query,
        location=location,
        remote_only=remote_only,
        top_n=int(top_n),
    )

    _state["result"] = result

    if not result.scored_jobs:
        jobs_msg = f"Pipeline ran. Jobs scraped: {result.jobs_found}. "
        if result.jobs_found == 0:
            jobs_msg += "All scrapers returned 0 results — check your internet connection or try a simpler query."
        else:
            jobs_msg += "Jobs were found but none scored above threshold. Try a broader query."
        return (
            gr.update(value=jobs_msg, visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            "",
            gr.update(interactive=True, value="Find & Score Jobs"),
        )

    import pandas as pd
    df = pd.DataFrame([
        {
            "Rank":     i + 1,
            "Score":    s.score,
            "Title":    s.job.title,
            "Company":  s.job.company,
            "Location": s.job.location,
            "Salary":   s.job.salary or "—",
            "Source":   s.job.source,
            "URL":      s.job.url,
        }
        for i, s in enumerate(result.scored_jobs)
    ])

    job_choices = [
        f"{i+1}. {s.job.title} @ {s.job.company} (Score: {s.score})"
        for i, s in enumerate(result.scored_jobs)
    ]

    profile = result.profile
    profile_md = (
        f"**Name:** {profile.name}  \n"
        f"**Email:** {profile.email}  \n"
        f"**Skills extracted:** {', '.join(profile.skills[:12])}  \n"
        f"**Jobs found:** {result.jobs_found} → Top {len(result.scored_jobs)} scored"
    )

    return (
        gr.update(value=profile_md, visible=True),
        gr.update(value=df, visible=True),
        gr.update(choices=job_choices, value=job_choices[0], visible=True),
        gr.update(visible=True),
        "",
        gr.update(interactive=True, value="Find & Score Jobs"),
    )


def _show_letter(job_choice):
    result = _state.get("result")
    if not result or not job_choice:
        return "", gr.update(visible=False)
    idx = int(job_choice.split(".")[0]) - 1
    scored = result.scored_jobs[idx]
    text = result.cover_letters.get(scored.job.uid, "")
    detail = (
        f"**Match Score:** {scored.score}/100  \n"
        f"**Reasons:** {' | '.join(scored.reasons)}  \n"
        f"**Skill gaps:** {', '.join(scored.skill_gaps) or 'None identified'}  \n"
        f"**Recommendation:** {scored.recommendation}  \n\n---\n\n"
        f"{text}"
    )
    return detail, gr.update(visible=True)


def _prepare_download(job_choice):
    result = _state.get("result")
    if not result or not job_choice:
        return gr.update(visible=False)
    idx = int(job_choice.split(".")[0]) - 1
    scored = result.scored_jobs[idx]
    docx_bytes = result.cover_letter_docs.get(scored.job.uid, b"")
    safe_name = scored.job.company.replace(" ", "_").replace("/", "-")
    import tempfile
    out = Path(tempfile.gettempdir()) / f"cover_letter_{safe_name}.docx"
    out.write_bytes(docx_bytes)
    return gr.update(value=str(out), visible=True)


with gr.Blocks(title="Job Search & Application System", theme=gr.themes.Soft()) as app:
    gr.Markdown("# Job Search & Application System")
    gr.Markdown(
        "Upload your resume, describe your target role, and receive "
        "scored job matches with tailored cover letters."
    )

    with gr.Row():
        with gr.Column(scale=1):
            resume_input   = gr.File(label="Resume (PDF)", file_types=[".pdf"])
            query_input    = gr.Textbox(label="Job Title / Keywords", placeholder="e.g. Machine Learning Engineer")
            location_input = gr.Textbox(label="Location", value="Remote")
            remote_toggle  = gr.Checkbox(label="Remote only", value=True)
            top_n_slider   = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Results to score")
            run_btn        = gr.Button("Find & Score Jobs", variant="primary")

        with gr.Column(scale=2):
            profile_md    = gr.Markdown(visible=False)
            jobs_table    = gr.Dataframe(visible=False, interactive=False)
            job_selector  = gr.Dropdown(label="Select a job to view cover letter", visible=False)
            view_btn      = gr.Button("View Cover Letter & Details", visible=False)
            letter_md     = gr.Markdown()
            download_file = gr.File(label="Cover Letter", visible=False, interactive=False)

    run_btn.click(
        fn=_set_loading,
        inputs=[],
        outputs=[profile_md, jobs_table, job_selector, view_btn, letter_md, run_btn],
        show_progress="hidden",
        queue=False,
    ).then(
        fn=_run,
        inputs=[resume_input, query_input, location_input, remote_toggle, top_n_slider],
        outputs=[profile_md, jobs_table, job_selector, view_btn, letter_md, run_btn],
        show_progress="full",
    )
    view_btn.click(
        fn=_show_letter,
        inputs=[job_selector],
        outputs=[letter_md, download_file],
        show_progress="hidden",
    )
    job_selector.change(
        fn=_prepare_download,
        inputs=[job_selector],
        outputs=[download_file],
        show_progress="hidden",
    )

app.launch(share=False)